In [1]:
# Install transformers and other necessary libraries
!pip install uv
!uv pip install transformers==4.28.0 transformer_lens torch numpy pandas matplotlib seaborn scipy tqdm nnsight

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 37.3 MB/s eta 0:00:00
Using Python 3.11.13 environment at: /usr
Resolved 117 packages in 3.59s
⠙ Preparing packages... (0/25)
⠙ Preparing packages... (0/25)
⠙ Preparing packages... (0/25)
⠙ Preparing packages... (0/25)
python-engineio          ------------------------------     0 B/58.14 KiB
⠙ Preparing packages... (0/25)
python-engineio          ------------------------------ 14.88 KiB/58.14 KiB
⠙ Preparing packages... (0/25)
python-engineio          ------------------------------ 14.88 KiB/58.14 KiB
⠙ Preparing packages... (0/25)
python-engineio          ------------------------------ 14.88 KiB/58.14 KiB
⠙ Preparing packages... (0/25)
jaxtyping                ------------------------------     0 B/54.11 KiB
python-engineio          ------------------------------ 14.88 KiB/58.14 KiB
⠙ Preparing packages... (0/25)
jaxtyping                ------------------------------     0 B/54.11 KiB
python-engineio          ------------------

In [2]:
#!/usr/bin/env python3
"""
Optimized N-gram Dataset Builder - Much Faster Version
"""

import torch
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Any, Optional
import matplotlib.pyplot as plt
import pickle
import re
import json
import time
import random
from tqdm import tqdm
import requests
from datasets import Dataset, load_dataset
import transformers
from transformers import AutoTokenizer
from pathlib import Path
import concurrent.futures
import threading

# Reduced frequency bins for faster processing
FREQUENCY_BINS = {
    'high': {'range': (1000, float('inf')), 'target_count': 200, 'description': 'Common phrases'},
    'medium': {'range': (100, 1000), 'target_count': 300, 'description': 'Uncommon phrases'},
    'low': {'range': (10, 100), 'target_count': 200, 'description': 'Rare phrases'},
    'very_low': {'range': (1, 10), 'target_count': 100, 'description': 'Very rare phrases'}
}

class OptimizedNgramBuilder:
    def __init__(self, cache_dir: str = "./ngram_cache", max_workers: int = 5):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.frequency_cache = self.load_frequency_cache()
        self.max_workers = max_workers
        self.rate_limit_delay = 0.05  # Reduced delay
        self.api_base_url = 'https://api.infini-gram.io/'
        self.session = requests.Session()  # Reuse connections

    def load_frequency_cache(self) -> Dict[str, int]:
        """Load cached frequency data"""
        cache_file = self.cache_dir / "frequency_cache.json"
        if cache_file.exists():
            with open(cache_file, 'r') as f:
                return json.load(f)
        return {}

    def save_frequency_cache(self):
        """Save frequency cache to disk"""
        cache_file = self.cache_dir / "frequency_cache.json"
        with open(cache_file, 'w') as f:
            json.dump(self.frequency_cache, f)

    def query_infinigram_batch(self, ngrams: List[str], corpus: str = "pile") -> Dict[str, int]:
        """Query multiple n-grams with threading for better performance"""
        results = {}
        lock = threading.Lock()

        def query_single(ngram: str):
            cache_key = f"{ngram}_{corpus}"

            # Check cache first
            with lock:
                if cache_key in self.frequency_cache:
                    results[ngram] = self.frequency_cache[cache_key]
                    return

            try:
                payload = {
                    'index': 'v4_piletrain_llama',
                    'query_type': 'count',
                    'query': ngram
                }

                response = self.session.post(self.api_base_url, json=payload, timeout=5)

                if response.status_code == 200:
                    result = response.json()
                    frequency = result.get('count', 0)

                    with lock:
                        self.frequency_cache[cache_key] = frequency
                        results[ngram] = frequency
                else:
                    print(f"API error for '{ngram}': {response.status_code}")
                    with lock:
                        results[ngram] = 0

            except Exception as e:
                print(f"Error querying '{ngram}': {e}")
                with lock:
                    results[ngram] = 0

            # Rate limiting
            time.sleep(self.rate_limit_delay)

        # Use ThreadPoolExecutor for concurrent requests
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [executor.submit(query_single, ngram) for ngram in ngrams]
            concurrent.futures.wait(futures)

        return results

    def extract_ngrams_from_text(self, text: str, n_gram_size: int) -> List[str]:
        """Extract n-grams from text - optimized version"""
        # More aggressive text cleaning
        text = re.sub(r'[^\w\s]', ' ', text.lower())  # Remove punctuation, lowercase
        text = re.sub(r'\s+', ' ', text.strip())
        words = text.split()

        if len(words) < n_gram_size:
            return []

        # Use list comprehension for speed
        return [' '.join(words[i:i + n_gram_size]) for i in range(len(words) - n_gram_size + 1)]

    def load_pile_dataset_fast(self, num_samples: int = 1000) -> List[str]:
        """Load samples from The Pile dataset - faster version"""
        print(f"Loading {num_samples} samples from The Pile...")

        try:
            dataset = load_dataset(
                "EleutherAI/pile",           # dataset identifier
                "hacker_news",               # choose any valid subset (e.g. 'all', 'enron_emails', …)
                split="train",
                streaming=True,              # stream instead of download
            )

            texts = []
            for i, sample in enumerate(dataset):
                if i >= num_samples:
                    break

                text = sample.get('text', '')
                if 50 < len(text.strip()) < 2000:  # Filter very short/long texts
                    texts.append(text.strip())

                if i % 100 == 0:
                    print(f"Loaded {len(texts)} valid texts from {i+1} samples")

            print(f"Final: {len(texts)} valid text samples from {num_samples} total")
            return texts

        except Exception as e:
            print(f"Error loading dataset: {e}")
            return self.generate_dummy_texts(num_samples)

    def generate_dummy_texts(self, num_samples: int = 100) -> List[str]:
        """Generate dummy texts for testing"""
        templates = [
            "The {adj} {noun} {verb} quickly in the {place}.",
            "Machine learning {verb} {adj} patterns in {noun} data.",
            "When {noun} {verb}, the {adj} system becomes {adj2}.",
            "The {adj} algorithm {verb} {noun} with {adj2} accuracy.",
            "Deep learning {verb} {adj} features from {noun} datasets."
        ]

        words = {
            'adj': ["advanced", "complex", "efficient", "robust", "scalable", "optimal"],
            'adj2': ["better", "faster", "stronger", "smarter", "cleaner", "simpler"],
            'noun': ["model", "system", "network", "algorithm", "data", "method"],
            'verb': ["processes", "analyzes", "computes", "learns", "predicts", "optimizes"],
            'place': ["cloud", "server", "database", "memory", "cache", "pipeline"]
        }

        texts = []
        for _ in range(num_samples):
            template = random.choice(templates)
            text = template.format(**{k: random.choice(v) for k, v in words.items()})
            texts.append(text)

        return texts

    def count_ngrams_fast(self, texts: List[str], n_gram_size: int) -> Tuple[Counter, Dict[str, List[str]]]:
        """Fast n-gram counting with filtering"""
        ngram_counts = Counter()
        ngram_text_mapping = defaultdict(list)

        print(f"Extracting {n_gram_size}-grams from {len(texts)} texts...")

        for text in tqdm(texts, desc="Processing texts"):
            ngrams = self.extract_ngrams_from_text(text, n_gram_size)

            for ngram in ngrams:
                # Basic quality filtering
                words = ngram.split()
                if (len(words) == n_gram_size and
                    all(len(word) >= 2 for word in words) and
                    not any(word.isdigit() for word in words)):

                    ngram_counts[ngram] += 1
                    if len(ngram_text_mapping[ngram]) < 3:  # Limit examples
                        ngram_text_mapping[ngram].append(text[:200])  # Truncate text

        print(f"Found {len(ngram_counts)} unique n-grams")
        return ngram_counts, dict(ngram_text_mapping)

    def smart_stratify_ngrams(self, ngram_counts: Counter, text_mapping: Dict[str, List[str]],
                             max_api_calls: int = 500) -> Dict[str, Dict[str, List]]:
        """Smart stratification with selective API calls"""

        # Pre-filter n-grams: only query promising candidates
        sorted_ngrams = ngram_counts.most_common()

        # Smart selection: query high-frequency local n-grams first (more likely to have global data)
        candidates_to_query = []
        for ngram, local_freq in sorted_ngrams:
            if (local_freq >= 3 and  # Must appear multiple times locally
                len(candidates_to_query) < max_api_calls):
                candidates_to_query.append(ngram)

        print(f"Querying {len(candidates_to_query)} selected n-grams (out of {len(sorted_ngrams)} total)")

        # Batch query in chunks
        chunk_size = 50
        global_frequencies = {}

        for i in range(0, len(candidates_to_query), chunk_size):
            chunk = candidates_to_query[i:i + chunk_size]
            print(f"Processing batch {i//chunk_size + 1}/{(len(candidates_to_query)-1)//chunk_size + 1}")

            chunk_results = self.query_infinigram_batch(chunk)
            global_frequencies.update(chunk_results)

            # Save cache periodically
            if i % (chunk_size * 5) == 0:
                self.save_frequency_cache()

        # Initialize stratified data
        stratified_data = {
            bin_name: {
                'ngrams': [], 'local_frequencies': [], 'global_frequencies': [],
                'text_examples': [], 'confidence_scores': []
            } for bin_name in FREQUENCY_BINS.keys()
        }

        # Categorize all n-grams (queried + estimated)
        for ngram, local_freq in sorted_ngrams:
            global_freq = global_frequencies.get(ngram)

            # If no global data, estimate based on local frequency
            if global_freq is None:
                global_freq = max(1, int(local_freq * random.uniform(50, 200)))  # Rough estimate

            # Categorize
            category = self.categorize_by_frequency(global_freq)

            # Add to appropriate bin if there's space
            if len(stratified_data[category]['ngrams']) < FREQUENCY_BINS[category]['target_count']:
                confidence = self.calculate_confidence_score(ngram, local_freq,
                                                           global_frequencies.get(ngram))

                stratified_data[category]['ngrams'].append(ngram)
                stratified_data[category]['local_frequencies'].append(local_freq)
                stratified_data[category]['global_frequencies'].append(global_freq)
                stratified_data[category]['text_examples'].append(text_mapping.get(ngram, [])[:2])
                stratified_data[category]['confidence_scores'].append(confidence)

        self.save_frequency_cache()
        return stratified_data

    def categorize_by_frequency(self, frequency: int) -> str:
        """Categorize n-gram by frequency"""
        for category, info in FREQUENCY_BINS.items():
            min_freq, max_freq = info['range']
            if min_freq <= frequency < max_freq:
                return category
        return 'very_low'

    def calculate_confidence_score(self, ngram: str, local_freq: int, global_freq: Optional[int]) -> float:
        """Calculate confidence score"""
        confidence = 0.8  # Base confidence

        # Boost for longer average word length
        avg_word_length = np.mean([len(word) for word in ngram.split()])
        if avg_word_length >= 4:
            confidence += 0.1

        # Boost if we have real global frequency data
        if global_freq is not None:
            confidence += 0.1

        # Boost for higher local frequency
        if local_freq >= 5:
            confidence += 0.05

        return min(confidence, 1.0)

    def build_fast_dataset(self, n_gram_size: int = 2, pile_samples: int = 1000,
                          max_api_calls: int = 300) -> Dict[str, Any]:
        """Build dataset with optimizations for speed"""

        print(f"🚀 Building FAST {n_gram_size}-gram dataset with {pile_samples} samples")
        start_time = time.time()

        # Step 1: Load texts (reduced samples for speed)
        texts = self.load_pile_dataset_fast(pile_samples)

        # Step 2: Extract and count n-grams
        ngram_counts, text_mapping = self.count_ngrams_fast(texts, n_gram_size)

        # Step 3: Smart stratification with limited API calls
        stratified_data = self.smart_stratify_ngrams(ngram_counts, text_mapping, max_api_calls)

        # Step 4: Create final dataset
        final_dataset = self.create_final_dataset(stratified_data, n_gram_size, pile_samples)

        elapsed_time = time.time() - start_time
        print(f"✅ Dataset built in {elapsed_time:.1f} seconds!")

        return final_dataset

    def create_final_dataset(self, stratified_data: Dict, n_gram_size: int, pile_samples: int) -> Dict[str, Any]:
        """Create final dataset structure"""

        final_dataset = {
            'texts': [], 'ngrams': [], 'categories': [],
            'local_frequencies': [], 'global_frequencies': [],
            'confidence_scores': [], 'metadata': []
        }

        for category, data in stratified_data.items():
            for i, (ngram, local_freq, global_freq, text_examples, confidence) in enumerate(zip(
                data['ngrams'], data['local_frequencies'], data['global_frequencies'],
                data['text_examples'], data['confidence_scores']
            )):
                for text_example in text_examples[:1]:  # Just 1 example per n-gram for speed
                    final_dataset['texts'].append(text_example)
                    final_dataset['ngrams'].append(ngram)
                    final_dataset['categories'].append(category)
                    final_dataset['local_frequencies'].append(local_freq)
                    final_dataset['global_frequencies'].append(global_freq)
                    final_dataset['confidence_scores'].append(confidence)
                    final_dataset['metadata'].append({'category': category, 'rank': i})

        final_dataset['dataset_metadata'] = {
            'n_gram_size': n_gram_size,
            'pile_samples': pile_samples,
            'total_samples': len(final_dataset['texts']),
            'creation_timestamp': time.time(),
            'frequency_bins': FREQUENCY_BINS
        }

        return final_dataset


# Convenience functions
def build_fast_ngram_dataset(n_gram_size: int = 2, pile_samples: int = 500,
                           max_api_calls: int = 200) -> Dict[str, Any]:
    """Quick entry point for fast dataset building"""
    builder = OptimizedNgramBuilder(max_workers=3)  # Conservative threading
    return builder.build_fast_dataset(n_gram_size, pile_samples, max_api_calls)


def save_dataset(dataset: Dict[str, Any], output_path: str):
    """Save dataset to JSON"""
    output_path = Path(output_path).with_suffix('.json')
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, 'w') as f:
        json.dump(dataset, f, indent=2, default=str)
    print(f"Saved dataset to {output_path}")


# Example usage
if __name__ == "__main__":
    print("🔥 Fast N-gram Dataset Builder")

    # Build a small, fast dataset
    dataset = build_fast_ngram_dataset(
        n_gram_size=2,
        pile_samples=500,     # Reduced for speed
        max_api_calls=150     # Limited API calls
    )

    # Quick stats
    print(f"\n📊 Dataset Stats:")
    print(f"Total samples: {len(dataset['texts'])}")
    print(f"Unique n-grams: {len(set(dataset['ngrams']))}")

    categories = Counter(dataset['categories'])
    for cat, count in categories.items():
        print(f"{cat}: {count} samples")

    # Save
    save_dataset(dataset, "./fast_ngram_dataset")
    print("✅ Done!")
    print(dataset)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

🔥 Fast N-gram Dataset Builder
🚀 Building FAST 2-gram dataset with 500 samples
Loading 500 samples from The Pile...
Error loading dataset: Dataset scripts are no longer supported, but found pile.py
Extracting 2-grams from 500 texts...


Processing texts: 100%|██████████| 500/500 [00:00<00:00, 56343.25it/s]

Found 244 unique n-grams
Querying 150 selected n-grams (out of 244 total)
Processing batch 1/3
Processing batch 2/3
Processing batch 3/3


KeyboardInterrupt: 

In [ ]:
#!/usr/bin/env python3
"""
Optimized Checkpoint Analysis for Polytope Evolution
Key optimizations: model reuse, frequency tracking, batch tokenization
"""
import torch
import numpy as np
import pandas as pd
from nnsight import LanguageModel
from typing import List, Dict, Any, Tuple, Optional, Union
import re
from difflib import SequenceMatcher
import json
from tqdm import tqdm
from collections import defaultdict, Counter
import warnings
from pathlib import Path


def create_activation_record(checkpoint_step: str,
                           text_idx: int,
                           text: str,
                           ngram: str,
                           matched_text: str,
                           match_confidence: float,
                           matching_strategy: str,
                           category: str,
                           layer: int,
                           token_position: int,
                           char_start: int,
                           char_end: int,
                           activation_vector: np.ndarray,
                           ngram_frequency: int = 1,  # NEW: frequency tracking
                           neuron_idx: Optional[int] = None,
                           **kwargs) -> Dict[str, Any]:
    """Create a structured activation record as dictionary"""

    # Compute additional metrics
    binary_pattern = (activation_vector > 0).astype(int)
    sparsity = 1.0 - (np.count_nonzero(activation_vector) / len(activation_vector))
    activation_norm = float(np.linalg.norm(activation_vector))
    n_active_neurons = int(np.count_nonzero(activation_vector))

    return {
        'checkpoint_step': checkpoint_step,
        'text_idx': text_idx,
        'text': text,
        'ngram': ngram,
        'matched_text': matched_text,
        'match_confidence': match_confidence,
        'matching_strategy': matching_strategy,
        'category': category,
        'layer': layer,
        'neuron_idx': neuron_idx,
        'token_position': token_position,
        'char_start': char_start,
        'char_end': char_end,
        'activation_vector': activation_vector,
        'binary_pattern': binary_pattern,
        'sparsity': sparsity,
        'activation_norm': activation_norm,
        'n_active_neurons': n_active_neurons,
        'ngram_frequency': ngram_frequency,  # NEW: track frequency
        'metadata': kwargs
    }


def compute_ngram_frequencies(dataset: Dict[str, List[Any]]) -> Dict[str, int]:
    """Compute frequency of each n-gram in the dataset"""
    ngram_counts = Counter(dataset['ngrams'])
    return dict(ngram_counts)


def batch_tokenize_positions(model: LanguageModel, texts: List[str], char_positions: List[int]) -> List[int]:
    """Efficiently compute token positions for multiple texts"""
    token_positions = []

    for text, char_pos in zip(texts, char_positions):
        # Tokenize prefix up to character position
        prefix = text[:char_pos] if char_pos > 0 else ""
        if prefix:
            tokens = model.tokenizer(prefix, return_tensors="pt")
            token_pos = tokens['input_ids'].shape[1] - 1
        else:
            token_pos = 0
        token_positions.append(token_pos)

    return token_positions


def organize_records_by_key(records: List[Dict[str, Any]], key: str) -> Dict[Any, List[Dict[str, Any]]]:
    """Organize records by a specific key"""
    organized = defaultdict(list)
    for record in records:
        organized[record[key]].append(record)
    return dict(organized)


def get_activation_matrix(records: List[Dict[str, Any]]) -> np.ndarray:
    """Extract activation matrix from records"""
    if not records:
        return np.array([])
    return np.stack([r['activation_vector'] for r in records])


def convert_records_to_dataframe(records: List[Dict[str, Any]]) -> pd.DataFrame:
    """Convert records to pandas DataFrame"""
    if not records:
        return pd.DataFrame()

    # Extract non-array fields for DataFrame
    df_data = []
    for record in records:
        row = {k: v for k, v in record.items()
               if k not in ['activation_vector', 'binary_pattern', 'metadata']}
        # Add select metadata fields
        if 'metadata' in record:
            for meta_key, meta_val in record['metadata'].items():
                row[f'meta_{meta_key}'] = meta_val
        df_data.append(row)

    return pd.DataFrame(df_data)


def find_ngram_matches(text: str, ngram: str, strategy: str = "exact") -> List[Tuple[int, int, float, str]]:
    """
    Find all occurrences of an n-gram in text with confidence scores
    OPTIMIZED: Default to exact matching for speed
    """
    if strategy == "exact":
        return _find_exact_matches(text, ngram)
    elif strategy == "fuzzy":
        return _find_fuzzy_matches(text, ngram)
    elif strategy == "comprehensive":
        return _find_comprehensive_matches(text, ngram)
    else:
        return _find_exact_matches(text, ngram)


def _find_exact_matches(text: str, ngram: str) -> List[Tuple[int, int, float, str]]:
    """Find exact string matches"""
    matches = []
    start = 0
    while True:
        pos = text.find(ngram, start)
        if pos == -1:
            break
        matches.append((pos, pos + len(ngram), 1.0, ngram))
        start = pos + 1
    return matches


def _find_fuzzy_matches(text: str, ngram: str, threshold: float = 0.8) -> List[Tuple[int, int, float, str]]:
    """Find fuzzy matches using sequence similarity"""
    matches = []
    ngram_len = len(ngram)

    for i in range(len(text) - ngram_len + 1):
        window = text[i:i + ngram_len]
        similarity = SequenceMatcher(None, ngram.lower(), window.lower()).ratio()

        if similarity >= threshold:
            confidence = similarity * 0.8  # Lower confidence for fuzzy matches
            matches.append((i, i + ngram_len, confidence, window))

    return matches


def _find_comprehensive_matches(text: str, ngram: str) -> List[Tuple[int, int, float, str]]:
    """Find matches using multiple strategies"""
    all_matches = []

    # Strategy 1: Exact matches
    exact_matches = _find_exact_matches(text, ngram)
    all_matches.extend([(start, end, conf, matched) for start, end, conf, matched in exact_matches])

    # Strategy 2: Case insensitive
    for match in re.finditer(re.escape(ngram), text, re.IGNORECASE):
        all_matches.append((match.start(), match.end(), 0.9, match.group()))

    # Strategy 3: Word boundaries
    escaped_ngram = re.escape(ngram)
    for pattern, confidence in [(rf'\b{escaped_ngram}\b', 0.95), (rf'\b{escaped_ngram}', 0.7)]:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            all_matches.append((match.start(), match.end(), confidence, match.group()))

    # Remove duplicates and sort by confidence
    unique_matches = []
    seen_positions = set()

    for start, end, conf, matched in sorted(all_matches, key=lambda x: x[2], reverse=True):
        pos_key = (start, end)
        if pos_key not in seen_positions:
            unique_matches.append((start, end, conf, matched))
            seen_positions.add(pos_key)

    return unique_matches[:3]  # OPTIMIZED: Return top 3 instead of 5


def extract_layer_activations_batch(model: LanguageModel,
                                  texts: List[str],
                                  layer: int,
                                  positions: List[int]) -> List[np.ndarray]:
    """
    OPTIMIZED: Extract activations for multiple texts in a single model pass
    """
    activations = []

    for text, position in zip(texts, positions):
        try:
            with model.trace(text) as tracer:
                mlp_post_activation = model.gpt_neox.layers[layer].mlp.act.output
                
                if position == -1:
                    position = mlp_post_activation.shape[1] - 1

                activation_tensor = mlp_post_activation[0, position, :].save()

            activation_np = activation_tensor.cpu().detach().numpy()
            activations.append(activation_np)

        except Exception as e:
            raise e

    return activations


def extract_activations_from_dataset(model_name: str,
                                   checkpoints: List[str],
                                   dataset: Dict[str, List[Any]],
                                   target_layers: List[int] = None,
                                   batch_size: int = 8) -> List[Dict[str, Any]]:
    if target_layers is None:
        target_layers = [0, 2, 4]

    all_records = []
    total_texts = len(dataset['texts'])

    print(f"Extracting activations for {len(checkpoints)} checkpoints")
    print(f"Dataset size: {total_texts} samples")
    print(f"Target layers: {target_layers}")

    for checkpoint_idx, checkpoint in enumerate(checkpoints):
        print(f"\nProcessing checkpoint {checkpoint_idx+1}/{len(checkpoints)}: step{checkpoint}")

        # Load model
        try:
            model = LanguageModel(model_name, revision=f"step{checkpoint}")
            print(f"Loaded model for checkpoint {checkpoint}")
        except Exception as e:
            warnings.warn(f"Error loading model {model_name} at step {checkpoint}: {e}")
            continue

        checkpoint_records = []

        for batch_start in tqdm(range(0, total_texts, batch_size), desc=f"Checkpoint {checkpoint}"):
            batch_end = min(batch_start + batch_size, total_texts)

            batch_texts = dataset['texts'][batch_start:batch_end]
            batch_ngrams = dataset['ngrams'][batch_start:batch_end]
            batch_categories = dataset.get('categories', ['unknown'] * total_texts)[batch_start:batch_end]
            batch_frequencies = dataset.get('local_frequencies', [1] * total_texts)[batch_start:batch_end]

            batch_matches = []
            batch_positions = []

            for text, ngram in zip(batch_texts, batch_ngrams):
                matches = find_ngram_matches(text, ngram, strategy="comprehensive")
                if matches:
                    char_start, char_end, confidence, matched_text = matches[0]
                    batch_matches.append((char_start, char_end, confidence, matched_text))
                    batch_positions.append(char_start)
                else:
                    batch_matches.append(None)
                    batch_positions.append(0)

            valid_texts = [text for text, match in zip(batch_texts, batch_matches) if match is not None]
            valid_positions = [pos for pos, match in zip(batch_positions, batch_matches) if match is not None]

            if valid_texts:
                token_positions = batch_tokenize_positions(model, valid_texts, valid_positions)
            else:
                token_positions = []

            for layer in target_layers:
                if valid_texts:
                    try:
                        layer_activations = extract_layer_activations_batch(
                            model, valid_texts, layer, token_positions
                        )

                        valid_idx = 0
                        for i in range(len(batch_texts)):
                            if batch_matches[i] is not None:
                                char_start, char_end, confidence, matched_text = batch_matches[i]
                                activation_vector = layer_activations[valid_idx]

                                if len(activation_vector) > 0:
                                    record = create_activation_record(
                                        checkpoint_step=checkpoint,
                                        text_idx=batch_start + i,
                                        text=batch_texts[i],
                                        ngram=batch_ngrams[i],
                                        matched_text=matched_text,
                                        match_confidence=confidence,
                                        matching_strategy="comprehensive",
                                        category=batch_categories[i],
                                        layer=layer,
                                        token_position=token_positions[valid_idx],
                                        char_start=char_start,
                                        char_end=char_end,
                                        activation_vector=activation_vector,
                                        ngram_frequency=batch_frequencies[i]
                                    )
                                    checkpoint_records.append(record)

                                valid_idx += 1

                    except Exception as e:
                        warnings.warn(f"Error extracting layer {layer} for batch: {e}")
                        continue

            torch.cuda.empty_cache()

        print(f"Extracted {len(checkpoint_records)} activation records for checkpoint {checkpoint}")
        all_records.extend(checkpoint_records)

        del model
        torch.cuda.empty_cache()

    print(f"\nTotal activation records extracted: {len(all_records)}")
    return all_records



def analyze_activation_patterns(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    ENHANCED: Analyze patterns with frequency analysis for polytope studies
    """
    if not records:
        return {'error': 'No records provided'}

    df = convert_records_to_dataframe(records)

    # Basic statistics
    analysis = {
        'total_records': len(records),
        'unique_checkpoints': df['checkpoint_step'].nunique(),
        'unique_ngrams': df['ngram'].nunique(),
        'unique_layers': df['layer'].nunique(),
        'mean_sparsity': df['sparsity'].mean(),
        'std_sparsity': df['sparsity'].std(),
        'mean_activation_norm': df['activation_norm'].mean(),
        'std_activation_norm': df['activation_norm'].std(),
        'mean_active_neurons': df['n_active_neurons'].mean(),
        'std_active_neurons': df['n_active_neurons'].std()
    }

    # NEW: Frequency analysis for polytope studies
    if 'ngram_frequency' in df.columns:
        freq_analysis = {
            'frequency_range': (df['ngram_frequency'].min(), df['ngram_frequency'].max()),
            'mean_frequency': df['ngram_frequency'].mean(),
            'frequency_bins': df['ngram_frequency'].value_counts().sort_index().to_dict()
        }
        analysis['frequency_analysis'] = freq_analysis

        # Sparsity vs frequency correlation
        if len(df) > 1:
            freq_sparsity_corr = df['ngram_frequency'].corr(df['sparsity'])
            analysis['frequency_sparsity_correlation'] = freq_sparsity_corr

    # Layer-wise analysis
    layer_analysis = {}
    for layer in df['layer'].unique():
        layer_df = df[df['layer'] == layer]
        layer_analysis[layer] = {
            'n_records': len(layer_df),
            'mean_sparsity': layer_df['sparsity'].mean(),
            'mean_norm': layer_df['activation_norm'].mean(),
            'mean_active': layer_df['n_active_neurons'].mean()
        }

    analysis['layer_analysis'] = layer_analysis

    # Category analysis
    if 'category' in df.columns:
        category_analysis = {}
        for category in df['category'].unique():
            cat_df = df[df['category'] == category]
            category_analysis[category] = {
                'n_records': len(cat_df),
                'mean_sparsity': cat_df['sparsity'].mean(),
                'mean_norm': cat_df['activation_norm'].mean()
            }
        analysis['category_analysis'] = category_analysis

    return analysis


# Keep existing helper functions unchanged
def save_activation_records(records: List[Dict[str, Any]],
                          output_path: str,
                          format: str = "pickle") -> None:
    """Save activation records to file"""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if format == "pickle":
        import pickle
        with open(output_path.with_suffix('.pkl'), 'wb') as f:
            pickle.dump(records, f)

    elif format == "csv":
        df = convert_records_to_dataframe(records)
        df.to_csv(output_path.with_suffix('.csv'), index=False)

    elif format == "json":
        json_records = []
        for record in records:
            json_record = record.copy()
            json_record['activation_vector'] = record['activation_vector'].tolist()
            json_record['binary_pattern'] = record['binary_pattern'].tolist()
            json_records.append(json_record)

        with open(output_path.with_suffix('.json'), 'w') as f:
            json.dump(json_records, f, indent=2)

    print(f"Saved {len(records)} records to {output_path}")


def load_activation_records(input_path: str) -> List[Dict[str, Any]]:
    """Load activation records from file"""
    input_path = Path(input_path)

    if input_path.suffix == '.pkl':
        import pickle
        with open(input_path, 'rb') as f:
            return pickle.load(f)

    elif input_path.suffix == '.json':
        with open(input_path, 'r') as f:
            json_records = json.load(f)

        for record in json_records:
            record['activation_vector'] = np.array(record['activation_vector'])
            record['binary_pattern'] = np.array(record['binary_pattern'])

        return json_records

    else:
        raise ValueError(f"Unsupported file format: {input_path.suffix}")




def main():
    """Example usage of optimized checkpoint analysis"""
    # Example data
    model_name = "EleutherAI/pythia-70m"
    checkpoints=['0', '1', '512', '1000', '10000', '50000', '143000']
    import json
    dataset = json.load(open('/content/fast_ngram_dataset.json'))

    records = extract_activations_from_dataset(
        model_name=model_name,
        checkpoints=checkpoints,
        dataset=dataset,
        target_layers=[4]
    )

    print(f"Extracted {len(records)} activation records")

    # Analyze patterns
    analysis = analyze_activation_patterns(records)
    print("\nActivation Analysis:")
    print(analysis)

    return records


if __name__ == "__main__":
    records = main()

Extracting activations for 7 checkpoints
Dataset size: 244 samples
Target layers: [4]

Processing checkpoint 1/7: step0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Using pad_token, but it is not set yet.


Loaded model for checkpoint 0


Checkpoint 0:   0%|          | 0/31 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/166M [00:00<?, ?B/s]

...

Checkpoint 143000:   0%|          | 0/31 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/166M [00:00<?, ?B/s]

Checkpoint 143000: 100%|██████████| 31/31 [00:15<00:00,  2.05it/s]

Extracted 238 activation records for checkpoint 143000

Total activation records extracted: 1666
Extracted 1666 activation records

Activation Analysis:
{'total_records': 1666, 'unique_checkpoints': 7, 'unique_ngrams': 238, 'unique_layers': 1, 'mean_sparsity': np.float64(0.0), 'std_sparsity': 0.0, 'mean_activation_norm': np.float64(25.103270320617565), 'std_activation_norm': 21.37705385996888, 'mean_active_neurons': np.float64(512.0), 'std_active_neurons': 0.0, 'frequency_analysis': {'frequency_range': (1, 103), 'mean_frequency': np.float64(14.27310924369748), 'frequency_bins': {1: 91, 2: 175, 3: 105, 4: 119, 5: 140, 6: 84, 7: 49, 8: 42, 9: 28, 10: 28, 11: 7, 12: 35, 13: 56, 14: 42, 15: 91, 16: 84, 17: 49, 18: 42, 19: 77, 20: 63, 21: 28, 22: 63, 23: 14, 24: 14, 26: 21, 35: 14, 44: 7, 45: 7, 46: 14, 47: 14, 49: 7, 69: 7, 95: 14, 101: 28, 103: 7}}, 'frequency_sparsity_correlation': np.float64(nan), 'layer_analysis': {np.int64(4): {'n_records': 1666, 'mean_sparsity': np.float64(0.0), 'mea


/usr/local/lib/python3.11/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.11/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [ ]:
#!/usr/bin/env python3
"""
Simple Polytope Analysis Functions for N-gram Activation Records
Research-focused implementation for studying superposition and frequency relationships
"""

import numpy as np
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings


def filter_activations_by_threshold(records: List[Dict[str, Any]],
                                  threshold: float = 0.1,
                                  layer: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Filter activation records by activation threshold to reduce noise

    Args:
        records: List of activation records
        threshold: Minimum activation norm threshold
        layer: Optional specific layer to filter (None for all layers)

    Returns:
        Filtered list of activation records
    """
    if layer is not None:
        records = [r for r in records if r['layer'] == layer]

    if not records:
        print("No records found after layer filtering.")
        return []

    # Get activation norms
    norms = np.array([r['activation_norm'] for r in records])

    # Compute quantile threshold
    threshold = np.quantile(norms, 0.75)

    # Filter records
    filtered_records = [r for r in records if r['activation_norm'] >= threshold]

    print(f"Quantile threshold at {0.9*100:.1f}th percentile: {threshold:.4f}")
    print(f"Filtered {len(records)} -> {len(filtered_records)} records")

    return filtered_records


def process_records_by_layer(records: List[Dict[str, Any]],
                           threshold: float = 0.1) -> Dict[int, List[Dict[str, Any]]]:
    """
    Process records layer by layer with threshold filtering

    Args:
        records: List of activation records
        threshold: Activation threshold for filtering

    Returns:
        Dictionary mapping layer -> filtered records
    """
    layer_records = defaultdict(list)

    # Group by layer first
    for record in records:
        layer_records[record['layer']].append(record)

    # Filter each layer by threshold
    filtered_layer_records = {}
    for layer, layer_recs in layer_records.items():
        filtered_recs = filter_activations_by_threshold(layer_recs, threshold, layer)
        if filtered_recs:  # Only keep layers with records above threshold
            filtered_layer_records[layer] = filtered_recs

    print(f"Processing {len(filtered_layer_records)} layers with threshold {threshold}")

    return filtered_layer_records


def compute_lower_dimensional_subspace(activation_matrix: np.ndarray,
                                     n_components: int = 10,
                                     method: str = "pca") -> Tuple[np.ndarray, Any]:
    """
    Compute lower dimensional representation of activation space

    Args:
        activation_matrix: Matrix of shape (n_samples, n_features)
        n_components: Number of dimensions to reduce to
        method: Dimensionality reduction method ('pca' only for now)

    Returns:
        (reduced_matrix, fitted_transformer)
    """
    if activation_matrix.shape[0] < 2:
        return activation_matrix, None

    if method == "pca":
        # Standardize first
        scaler = StandardScaler()
        scaled_matrix = scaler.fit_transform(activation_matrix)

        # Apply PCA
        n_components = min(n_components, activation_matrix.shape[0] - 1, activation_matrix.shape[1])
        pca = PCA(n_components=0.95)
        reduced_matrix = pca.fit_transform(scaled_matrix)

        print(f"PCA: {activation_matrix.shape} -> {reduced_matrix.shape}")
        print(f"Explained variance ratio: {pca.explained_variance_ratio_[:5]}")  # First 5 components

        return reduced_matrix, {'pca': pca, 'scaler': scaler}

    else:
        raise ValueError(f"Method {method} not supported")


def calculate_convex_hull_metrics(points: np.ndarray) -> Dict[str, float]:
    """
    Calculate convex hull and polytope metrics

    Args:
        points: Array of points in n-dimensional space

    Returns:
        Dictionary of polytope metrics
    """
    if points.shape[0] < points.shape[1] + 1:
        # Not enough points for convex hull in this dimension
        return {
            'volume': 0.0,
            'surface_area': 0.0,
            'n_vertices': points.shape[0],
            'n_facets': 0,
            'dimension': points.shape[1],
            'hull_valid': False
        }

    try:
        hull = ConvexHull(points)

        metrics = {
            'volume': hull.volume,
            'surface_area': hull.area,
            'n_vertices': len(hull.vertices),
            'n_facets': len(hull.simplices),
            'dimension': points.shape[1],
            'hull_valid': True
        }

        # Additional polytope metrics
        if points.shape[0] > 1:
            # Average pairwise distance (measure of spread)
            distances = pdist(points)
            metrics['mean_pairwise_distance'] = np.mean(distances)
            metrics['std_pairwise_distance'] = np.std(distances)

            # Centroid distance variance (measure of concentration)
            centroid = np.mean(points, axis=0)
            centroid_distances = np.linalg.norm(points - centroid, axis=1)
            metrics['centroid_variance'] = np.var(centroid_distances)

            # Effective dimension (participation ratio)
            # Higher values = more dimensions actively used
            point_norms = np.linalg.norm(points, axis=1)
            if np.sum(point_norms) > 0:
                normalized_norms = point_norms / np.sum(point_norms)
                effective_dim = 1.0 / np.sum(normalized_norms**2)
                metrics['effective_dimension'] = effective_dim
            else:
                metrics['effective_dimension'] = 0.0

        return metrics

    except Exception as e:
        warnings.warn(f"ConvexHull calculation failed: {e}")
        return {
            'volume': 0.0,
            'surface_area': 0.0,
            'n_vertices': points.shape[0],
            'n_facets': 0,
            'dimension': points.shape[1],
            'hull_valid': False,
            'mean_pairwise_distance': 0.0,
            'std_pairwise_distance': 0.0,
            'centroid_variance': 0.0,
            'effective_dimension': 0.0
        }


def analyze_layer_polytopes(layer_records: Dict[int, List[Dict[str, Any]]],
                          n_components: int = 20) -> Dict[int, Dict[str, Any]]:
    """
    Analyze polytope metrics for each layer

    Args:
        layer_records: Dictionary mapping layer -> list of records
        n_components: Number of PCA components for dimensionality reduction

    Returns:
        Dictionary mapping layer -> polytope analysis results
    """
    layer_analyses = {}

    for layer, records in layer_records.items():
        print(f"\nAnalyzing layer {layer} with {len(records)} records...")

        # Extract activation matrix
        activation_matrix = np.stack([r['activation_vector'] for r in records])

        # Reduce dimensionality
        reduced_matrix, transformer = compute_lower_dimensional_subspace(
            activation_matrix, n_components
        )

        # Calculate polytope metrics
        polytope_metrics = calculate_convex_hull_metrics(reduced_matrix)

        # Add layer-specific metrics
        analysis = {
            'layer': layer,
            'n_records': len(records),
            'original_dim': activation_matrix.shape[1],
            'reduced_dim': reduced_matrix.shape[1],
            'polytope_metrics': polytope_metrics,
            'transformer': transformer,
            'reduced_points': reduced_matrix
        }

        layer_analyses[layer] = analysis

    return layer_analyses


def group_records_by_frequency_bins(records: List[Dict[str, Any]],
                                  n_bins: int = 5) -> Dict[str, List[Dict[str, Any]]]:
    """
    Group records into frequency bins for analysis

    Args:
        records: List of activation records (must have 'ngram_frequency' field)
        n_bins: Number of frequency bins

    Returns:
        Dictionary mapping frequency_bin -> list of records
    """
    # Extract frequencies
    frequencies = [r['ngram_frequency'] for r in records if 'ngram_frequency' in r]

    if not frequencies:
        warnings.warn("No frequency information found in records")
        return {'unknown': records}

    # Create frequency bins
    freq_min, freq_max = min(frequencies), max(frequencies)
    bin_edges = np.linspace(freq_min, freq_max + 0.1, n_bins + 1)

    # Group records by bins
    frequency_groups = defaultdict(list)

    for record in records:
        if 'ngram_frequency' not in record:
            frequency_groups['unknown'].append(record)
            continue

        freq = record['ngram_frequency']
        bin_idx = np.digitize(freq, bin_edges) - 1
        bin_idx = max(0, min(bin_idx, n_bins - 1))  # Clamp to valid range

        bin_label = f"freq_{bin_edges[bin_idx]:.1f}-{bin_edges[bin_idx + 1]:.1f}"
        frequency_groups[bin_label].append(record)

    print(f"Created {len(frequency_groups)} frequency bins:")
    for bin_label, bin_records in frequency_groups.items():
        print(f"  {bin_label}: {len(bin_records)} records")

    return dict(frequency_groups)


def analyze_frequency_polytope_relationship(records: List[Dict[str, Any]],
                                          layers: List[int] = None,
                                          n_freq_bins: int = 5,
                                          n_components: int = 15) -> pd.DataFrame:
    """
    Analyze relationship between n-gram frequency and polytope metrics

    Args:
        records: List of activation records
        layers: List of layers to analyze (None for all)
        n_freq_bins: Number of frequency bins
        n_components: PCA components for analysis

    Returns:
        DataFrame with frequency-polytope analysis results
    """
    results = []

    # Group by frequency bins
    frequency_groups = group_records_by_frequency_bins(records, n_freq_bins)

    for freq_bin, freq_records in frequency_groups.items():
        if not freq_records:
            continue

        # Process by layer within each frequency bin
        layer_records = process_records_by_layer(freq_records, threshold=0.05)

        # Filter layers if specified
        if layers is not None:
            layer_records = {l: recs for l, recs in layer_records.items() if l in layers}

        # Analyze each layer
        for layer, recs in layer_records.items():
            if len(recs) < 3:  # Need minimum points for analysis
                continue

            # Extract activation matrix
            activation_matrix = np.stack([r['activation_vector'] for r in recs])

            # Reduce dimensionality
            reduced_matrix, _ = compute_lower_dimensional_subspace(
                activation_matrix, n_components
            )

            # Calculate polytope metrics
            polytope_metrics = calculate_convex_hull_metrics(reduced_matrix)

            # Compile result
            result = {
                'frequency_bin': freq_bin,
                'layer': layer,
                'n_samples': len(recs),
                'mean_frequency': np.mean([r['ngram_frequency'] for r in recs if 'ngram_frequency' in r]),
                **polytope_metrics
            }

            results.append(result)

    return pd.DataFrame(results)


def plot_polytope_frequency_relationship(analysis_df: pd.DataFrame,
                                       save_path: Optional[str] = None):
    """
    Plot relationship between polytope metrics and n-gram frequency

    Args:
        analysis_df: DataFrame from analyze_frequency_polytope_relationship
        save_path: Optional path to save the plot
    """
    if analysis_df.empty:
        print("No data to plot")
        return

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    # Metrics to plot
    metrics = ['volume', 'surface_area', 'effective_dimension',
              'mean_pairwise_distance', 'centroid_variance', 'n_vertices']

    for i, metric in enumerate(metrics):
        if i >= len(axes):
            break

        ax = axes[i]

        # Plot by layer
        for layer in sorted(analysis_df['layer'].unique()):
            layer_data = analysis_df[analysis_df['layer'] == layer]

            if len(layer_data) > 1:
                ax.scatter(layer_data['mean_frequency'], layer_data[metric],
                          label=f'Layer {layer}', alpha=0.7, s=50)

                # Add trend line
                if len(layer_data) > 2:
                    z = np.polyfit(layer_data['mean_frequency'], layer_data[metric], 1)
                    p = np.poly1d(z)
                    x_trend = np.linspace(layer_data['mean_frequency'].min(),
                                        layer_data['mean_frequency'].max(), 100)
                    ax.plot(x_trend, p(x_trend), '--', alpha=0.5)

        ax.set_xlabel('Mean N-gram Frequency')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(f'{metric.replace("_", " ").title()} vs Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.suptitle('Polytope Metrics vs N-gram Frequency by Layer',
                fontsize=16, y=1.02)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")

    plt.show()


def plot_checkpoint_evolution(records: List[Dict[str, Any]],
                            checkpoints: List[str],
                            layers: List[int] = None,
                            save_path: Optional[str] = None):
    """
    Plot evolution of polytope metrics across training checkpoints

    Args:
        records: List of activation records
        checkpoints: List of checkpoint steps
        layers: Layers to analyze (None for all)
        save_path: Optional path to save plot
    """
    results = []

    for checkpoint in checkpoints:
        checkpoint_records = [r for r in records if r['checkpoint_step'] == checkpoint]

        if not checkpoint_records:
            continue

        # Analyze this checkpoint
        freq_analysis = analyze_frequency_polytope_relationship(
            checkpoint_records, layers=layers, n_freq_bins=3
        )

        # Add checkpoint info
        freq_analysis['checkpoint'] = checkpoint
        results.append(freq_analysis)

    if not results:
        print("No data to plot")
        return

    combined_df = pd.concat(results, ignore_index=True)

    # Plot evolution
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()

    metrics = ['volume', 'effective_dimension', 'mean_pairwise_distance', 'centroid_variance']

    for i, metric in enumerate(metrics):
        ax = axes[i]

        # Plot by layer
        for layer in sorted(combined_df['layer'].unique()):
            layer_data = combined_df[combined_df['layer'] == layer]

            # Group by checkpoint and take mean
            checkpoint_means = layer_data.groupby('checkpoint')[metric].mean()

            if len(checkpoint_means) > 1:
                ax.plot(range(len(checkpoint_means)), checkpoint_means.values,
                       'o-', label=f'Layer {layer}', alpha=0.7)

        ax.set_xlabel('Training Checkpoint')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.set_title(f'{metric.replace("_", " ").title()} Evolution')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Set x-axis labels
        if len(checkpoints) <= 10:
            ax.set_xticks(range(len(checkpoints)))
            ax.set_xticklabels(checkpoints, rotation=45)

    plt.tight_layout()
    plt.suptitle('Polytope Evolution Across Training Checkpoints',
                fontsize=14, y=1.02)

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Evolution plot saved to {save_path}")

    plt.show()


# Example usage function
def run_polytope_analysis(records: List[Dict[str, Any]],
                        checkpoints: List[str] = None,
                        target_layers: List[int] = None) -> Dict[str, Any]:
    """
    Run complete polytope analysis pipeline

    Args:
        records: List of activation records
        checkpoints: List of checkpoints to analyze
        target_layers: List of layers to focus on

    Returns:
        Dictionary containing all analysis results
    """
    print("=== Running Polytope Analysis ===")

    # 1. Process records by layer with threshold
    layer_records = process_records_by_layer(records, threshold=0.1)

    if target_layers:
        layer_records = {l: recs for l, recs in layer_records.items() if l in target_layers}

    # 2. Analyze polytopes for each layer
    layer_analyses = analyze_layer_polytopes(layer_records)

    # 3. Frequency-polytope relationship analysis
    freq_analysis_df = analyze_frequency_polytope_relationship(records, target_layers)

    # 4. Plot relationships
    plot_polytope_frequency_relationship(freq_analysis_df)

    # 5. If multiple checkpoints, plot evolution
    if checkpoints and len(checkpoints) > 1:
        plot_checkpoint_evolution(records, checkpoints, target_layers)

    return {
        'layer_analyses': layer_analyses,
        'frequency_analysis': freq_analysis_df,
        'summary': {
            'n_layers_analyzed': len(layer_analyses),
            'n_frequency_bins': len(freq_analysis_df['frequency_bin'].unique()),
            'total_records': len(records)
        }
    }


if __name__ == "__main__":
    import json
    # Example usage would go here
    print("Polytope analysis functions loaded. Use run_polytope_analysis() to analyze your records.")
    results = run_polytope_analysis(
        records=records,
        checkpoints=['0', '1', '512', '1000', '10000', '50000', '143000'],
        target_layers=[4]
    )

Polytope analysis functions loaded. Use run_polytope_analysis() to analyze your records.
=== Running Polytope Analysis ===
Quantile threshold at 90.0th percentile: 41.1112
Filtered 1666 -> 167 records
Processing 1 layers with threshold 0.1

Analyzing layer 4 with 167 records...
PCA: (167, 512) -> (167, 31)
Explained variance ratio: [0.21055065 0.11996441 0.07941382 0.07304332 0.05917823]


/tmp/ipython-input-6-3532645007.py:180: UserWarning: ConvexHull calculation failed: QH6235 qhull error (qh_memalloc): negative request size (-49320184).  Did int overflow due to high-D?

While executing:  | qhull i Qt Qx
Options selected for Qhull 2020.2.r 2020/08/31:
  run-id 1051598095  incidence  Qtriangulate  Qxact-merge  _zero-centrum
  Q3-no-merge-vertices-dim-high  _max-width 37  Error-roundoff 8.8e-13
  _one-merge 5.6e-11  _near-inside 2.8e-10  Visible-distance 5.3e-12
  U-max-coplanar 5.3e-12  Width-outside 1.1e-11  _wide-facet 3.2e-11
  _narrow-hull 4.2e-15  _maxoutside 5.7e-11
Last point added to hull was p94.  Last merge was #66.

At error exit:

Convex hull of 167 points in 31-d:

  Number of vertices: 42
  Number of facets: 11353633
  Number of non-simplicial facets: 4

Statistics for:  | qhull i Qt Qx

  Number of points processed: 41
  Number of hyperplanes created: 6287882
  Number of distance tests for qhull: 6685876
  Number of distance tests for merging: 133719654
 

Created 5 frequency bins:
  freq_82.7-103.1: 49 records
  freq_62.3-82.7: 7 records
  freq_41.8-62.3: 49 records
  freq_21.4-41.8: 126 records
  freq_1.0-21.4: 1435 records
Quantile threshold at 90.0th percentile: 50.4156
Filtered 49 -> 5 records
Processing 1 layers with threshold 0.05
PCA: (5, 512) -> (5, 4)
Explained variance ratio: [0.37164825 0.28756225 0.2360915  0.10469805]
Quantile threshold at 90.0th percentile: 123.8889
Filtered 7 -> 1 records
Processing 1 layers with threshold 0.05
Quantile threshold at 90.0th percentile: 51.9633
Filtered 49 -> 5 records
Processing 1 layers with threshold 0.05
PCA: (5, 512) -> (5, 3)
Explained variance ratio: [0.49552917 0.26022446 0.24424629]
Quantile threshold at 90.0th percentile: 42.4894
Filtered 126 -> 13 records
Processing 1 layers with threshold 0.05
PCA: (13, 512) -> (13, 7)
Explained variance ratio: [0.33654505 0.2003017  0.15394492 0.11180218 0.08146945]
Quantile threshold at 90.0th percentile: 40.6977
Filtered 1435 -> 144 records
P

In [ ]:
# function to calculate convex hull for some activation vectors and get some polytope metrics
from scipy.spatial import ConvexHull
import numpy as np

def get_polytope_metrics(activation_vectors):
  """Calculate convex hull, polytope volume, faces nd vertices for complexity
      surface area"""
  convex_hull = ConvexHull(activation_vectors)
  polytope_volume = convex_hull.volume
  faces = convex_hull.simplices
  vertices = convex_hull.vertices
  return convex_hull, polytope_volume, faces, vertices


